# TRIBE v2 Demo: Predicting Brain Responses to Naturalistic Stimuli

[TRIBE v2](https://github.com/facebookresearch/tribev2) is a deep multimodal brain encoding model that predicts **fMRI brain responses** to naturalistic stimuli — video, audio, and text.

It combines state-of-the-art feature extractors — **LLaMA 3.2** (text), **V-JEPA2** (video), and **Wav2Vec-BERT** (audio) — into a unified Transformer that maps multimodal representations onto the cortical surface (**fsaverage5**, ~20k vertices).

In this notebook, we will:
1. Load a pretrained TRIBE v2 model from HuggingFace
2. Predict brain responses to a **video** clip
3. Predict brain responses to **audio** generated from text
4. Visualize the predicted activity on a 3D brain surface

## Setup (for Colab users)

1. Activate the GPU (Menu > Runtime > Change runtime)
2. Run the command below
3. Restart your environment for the new packages to be taken into account

In [4]:
!uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

Using Python 3.11.13 environment at: /usr
Resolved 134 packages in 1.25s
Audited 134 packages in 0.34ms


## Loading the model

We load TRIBE v2 model from [HuggingFace Hub](https://huggingface.co/facebook/tribev2). On the first run, this downloads the model checkpoint and config (~1 GB). Subsequent runs use the cached version.

We also initialize a `PlotBrain` object for 3D brain surface visualization using the **fsaverage5** mesh.

In [6]:
from tribev2.demo_utils import TribeModel, download_file
from tribev2.plotting import PlotBrain
from pathlib import Path

CACHE_FOLDER = Path("./cache")

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
)
plotter = PlotBrain(mesh="fsaverage5")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-07-27 23:31:58 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_miss

## Predict brain responses to a video

Given a video file, TRIBE v2 automatically:
1. **Extracts audio** from the video track
2. **Transcribes speech** into word-level events with timestamps using [**WhisperX**](https://github.com/m-bain/whisperx)
3. **Extracts visual features** (DINOv2 + V-JEPA2) and **audio features** (Wav2Vec-BERT) and **text features** (LLaMA 3.2)
4. **Predicts fMRI activity** at each time step (1 TR = 1 second) across the cortical surface

Below, we download a sample video ([Sintel trailer](https://durian.blender.org/)), build an events dataframe, and run the model.

In [4]:
video_path = CACHE_FOLDER / "sample_video.mp4"
url = "https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4"
download_file(url, video_path)
df = model.get_events_dataframe(video_path=video_path)
display(df.head(8)[["type", "start", "duration", "filepath", "text", "context"]])

INFO - Downloaded https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4 -> cache/sample_video.mp4
INFO:tribev2.demo_utils:Downloaded https://download.blender.org/durian/trailer/sintel_trailer-480p.mp4 -> cache/sample_video.mp4
Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 1662.43it/s]
/usr/local/lib/python3.11/dist-packages/neuralset/events/utils.py:134: UserWarning: The events dataframe contains an `Index` column. This is dangerous, please add drop=True in calls to df.reset_index(). Dropping it automatically.
  warnings.warn(msg)
Extracting words from audio:   0%|          | 0/1 [00:36<?, ?it/s]


RuntimeError: whisperx failed:
Traceback (most recent call last):
  File "/root/.cache/uv/archive-v0/P2dxvAJ76czXpORxxmo7V/bin/whisperx", line 12, in <module>
    sys.exit(cli())
             ^^^^^
  File "/root/.cache/uv/archive-v0/P2dxvAJ76czXpORxxmo7V/lib/python3.11/site-packages/whisperx/__main__.py", line 98, in cli
    transcribe_task(args, parser)
  File "/root/.cache/uv/archive-v0/P2dxvAJ76czXpORxxmo7V/lib/python3.11/site-packages/whisperx/transcribe.py", line 127, in transcribe_task
    model = load_model(
            ^^^^^^^^^^^
  File "/root/.cache/uv/archive-v0/P2dxvAJ76czXpORxxmo7V/lib/python3.11/site-packages/whisperx/asr.py", line 357, in load_model
    model = model or WhisperModel(whisper_arch,
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/.cache/uv/archive-v0/P2dxvAJ76czXpORxxmo7V/lib/python3.11/site-packages/faster_whisper/transcribe.py", line 689, in __init__
    self.model = ctranslate2.models.Whisper(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^
Va

### Run the model

We feed the events dataframe to `model.predict()`, which extracts features for each modality, runs them through the Transformer, and returns predicted brain activity.

NOTE: you will have to request access to the Llama-3.2 model using your HuggingFace account.

The output `preds` has shape `(n_timesteps, n_vertices)` — one prediction per second of stimulus, with ~20k cortical vertices. The `segments` list contains the corresponding time segments with their associated events.

In [5]:
preds, segments = model.predict(events=df)
print(f"Predictions shape: {preds.shape}  (n_timesteps, n_vertices)")


NameError: name 'df' is not defined

In [5]:
from huggingface_hub import notebook_login

notebook_login()

Después de ejecutar la celda anterior, el entorno de ejecución estará autenticado con Hugging Face. Ahora puedes intentar ejecutar la celda que causó el error (`preds, segments = model.predict(events=df)`).

### Visualize predictions on the brain surface

We plot the predicted fMRI activity for the first 15 time steps on the fsaverage5 cortical mesh. Each panel shows one second of predicted activity, with the corresponding stimulus frame displayed below. Predictions are offset by 5 seconds in the past, in order to compensate for the hemodynamic lag.

We see that as the image appears on the screen, the visual cortex lights up (t=4s), followed by the language network when the character starts to speak (t=12s).

In [ ]:
n_timesteps = 15
fig = plotter.plot_timesteps(preds[:n_timesteps], segments=segments[:n_timesteps], cmap="fire", norm_percentile=99, vmin=.6, alpha_cmap=(0, .2), show_stimuli=True)

## Predict brain responses to text (via text-to-speech)

TRIBE v2 can also predict brain responses to **text** input. Since the model was trained on naturalistic audio/video stimuli, text is first converted to speech using Google Text-to-Speech (gTTS), then transcribed back to obtain precise word-level timings.

Below, we use a passage from Shakespeare's *Hamlet* as input.

In [ ]:
text = """
To be or not to be, that is the question.
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die, to sleep,
No more; and by a sleep to say we end
The heartache and the thousand natural shocks
"""

text_path = CACHE_FOLDER / "shakespeare.txt"
text_path.write_text(text)

df = model.get_events_dataframe(text_path=text_path)
display(df.head(8)[["type", "start", "duration", "filepath", "text", "context"]])

### Run the model

Same as before — we pass the events dataframe to `model.predict()` to get brain activity predictions for each time step.

In [ ]:
preds, segments = model.predict(events=df)
print(f"Predictions shape: {preds.shape}  (n_timesteps, n_vertices)")

In [ ]:
# Visualiza los primeros 15 timesteps del array `preds`
# Puedes ajustar `n_timesteps` para visualizar más o menos datos.
n_timesteps = 15
fig = plotter.plot_timesteps(preds[:n_timesteps], segments=segments[:n_timesteps], cmap="magma", norm_percentile=99, vmin=.6, alpha_cmap=(0, .2), show_stimuli=False)


En la celda anterior:
- `preds[:n_timesteps]` selecciona los primeros `n_timesteps` de las predicciones.
- `segments[:n_timesteps]` proporciona la información de los segmentos de tiempo correspondientes.
- `cmap`, `norm_percentile`, `vmin`, `alpha_cmap` son parámetros de visualización para ajustar el mapa de color, la normalización y la transparencia.
- `show_stimuli=False` desactiva la visualización de los estímulos (video/audio) si no son relevantes para esta visualización específica.

El resultado mostrará una secuencia de cerebros 3D con la actividad predicha coloreada.

### Visualize predictions on the brain surface

Again, we visualize the first 15 seconds of predicted activity. For audio-only stimuli, the stimulus display shows the spoken words at each time step.

In [ ]:
n_timesteps = 15
fig = plotter.plot_timesteps(preds[:n_timesteps], segments=segments[:n_timesteps], cmap="fire", norm_percentile=99, vmin=.6, alpha_cmap=(0, .2), show_stimuli=True)

### 1. Verificar GPU

In [1]:
import subprocess

try:
    # Ejecuta el comando nvidia-smi y captura la salida
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True)
    print(result.stdout)
except FileNotFoundError:
    print("nvidia-smi no encontrado. ¿La GPU está activada y los drivers están instalados?")
except subprocess.CalledProcessError as e:
    print(f"Error al ejecutar nvidia-smi: {e.stderr}")


nvidia-smi no encontrado. ¿La GPU está activada y los drivers están instalados?


### 2. Aplicar Parche para WhisperX

Esta celda modificará el archivo `eventstransforms.py` dentro de la librería `tribev2` para añadir el argumento `--compute_type float32` a la llamada de `whisperx`. Esto debería resolver el error de `float16`.

**¡Importante!** Después de ejecutar esta celda, DEBES reiniciar el entorno de ejecución (Menú > Entorno de ejecución > Reiniciar entorno de ejecución) y luego volver a ejecutar TODAS las celdas desde el principio para que el cambio surta efecto.

In [2]:
# Encuentra la ruta del archivo eventstransforms.py de tribev2
import subprocess
import os

try:
    # Buscar el directorio base de la instalación de uv
    uv_base_path_output = subprocess.run(['uv', 'pip', 'list', '--format', 'json'], capture_output=True, text=True, check=True).stdout
    import json
    uv_packages = json.loads(uv_base_path_output)
    tribev2_path = None
    for pkg in uv_packages:
        if pkg['name'] == 'tribev2':
            # Asumiendo que la ruta está en 'location' o 'path'
            # Puede ser necesario ajustar si la estructura de uv pip list cambia
            tribev2_path = pkg.get('location') or pkg.get('path')
            break

    if not tribev2_path:
        raise FileNotFoundError("No se encontró la ruta de instalación de tribev2.")

    target_file_path = os.path.join(tribev2_path, 'tribev2', 'eventstransforms.py')

    if not os.path.exists(target_file_path):
        # Si la ruta anterior no funciona, intenta una búsqueda más general
        print(f"Buscando eventstransforms.py en /root/.cache/uv/archive-v0/")
        find_command = f"find /root/.cache/uv/archive-v0/ -name eventstransforms.py"
        result = subprocess.run(find_command, shell=True, capture_output=True, text=True, check=True)
        target_file_path = result.stdout.strip().split('\n')[0]
        if not target_file_path or not os.path.exists(target_file_path):
            raise FileNotFoundError(f"No se encontró el archivo eventstransforms.py. Buscado en {target_file_path} y con find.")

    print(f"Parcheando el archivo: {target_file_path}")

    # Usa sed para insertar '--compute_type', 'float32' después de '--language', language,
    # Asegúrate de escapar las barras (/) en el path si las hubiera en el patron de sed.
    # El patrón busca la línea que contiene '--language', language,
    # y la reemplaza añadiendo el nuevo argumento.
    sed_command = f"sed -i \"s|--language', language,|--language', language, '--compute_type', 'float32',|g\" {target_file_path}"

    subprocess.run(sed_command, shell=True, check=True)

    print("Parche aplicado exitosamente.")

except Exception as e:
    print(f"Ocurrió un error al aplicar el parche: {e}")
    print("Por favor, revisa que 'tribev2' esté correctamente instalado y que la ruta del archivo sea correcta.")


Ocurrió un error al aplicar el parche: No se encontró la ruta de instalación de tribev2.
Por favor, revisa que 'tribev2' esté correctamente instalado y que la ruta del archivo sea correcta.


### 2.1 Aplicar Parche para WhisperX (Versión Mejorada)

Esta celda modificará el archivo `eventstransforms.py` dentro de la librería `tribev2` para añadir el argumento `--compute_type float32` a la llamada de `whisperx`. Esta versión es más robusta para encontrar el archivo.

**¡Importante!** Después de ejecutar esta celda, DEBES reiniciar el entorno de ejecución (Menú > Entorno de ejecución > Reiniciar entorno de ejecución) y luego volver a ejecutar TODAS las celdas desde el principio para que el cambio surta efecto.

In [3]:
import tribev2
import os
import subprocess

try:
    # Usa __file__ para encontrar la ruta de instalación de tribev2
    tribev2_root = os.path.dirname(os.path.abspath(tribev2.__file__))
    target_file_path = os.path.join(tribev2_root, 'eventstransforms.py')

    if not os.path.exists(target_file_path):
        raise FileNotFoundError(f"No se encontró el archivo eventstransforms.py en {target_file_path}.")

    print(f"Parcheando el archivo: {target_file_path}")

    # Usa sed para insertar '--compute_type', 'float32' después de '--language', language,
    # El patrón busca la línea que contiene '--language', language,
    # y la reemplaza añadiendo el nuevo argumento.
    sed_command = f"sed -i \"s|--language', language,|--language', language, '--compute_type', 'float32',|g\" {target_file_path}"

    subprocess.run(sed_command, shell=True, check=True)

    print("Parche aplicado exitosamente.")

except Exception as e:
    print(f"Ocurrió un error al aplicar el parche: {e}")
    print("Por favor, revisa que 'tribev2' esté correctamente instalado y que la ruta del archivo sea correcta.")


/usr/local/lib/python3.11/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-07-27 23:31:16 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


Parcheando el archivo: /usr/local/lib/python3.11/dist-packages/tribev2/eventstransforms.py
Parche aplicado exitosamente.


### 3. Reiniciar y Re-ejecutar

**¡AHORA ES CRÍTICO!**

1.  Ve a `Menú > Entorno de ejecución > Reiniciar entorno de ejecución`.
2.  Después de reiniciar, **ejecuta TODAS las celdas desde el principio** (Menú > Entorno de ejecución > Ejecutar todo).

Esto asegurará que el parche se aplique y que el modelo `tribev2` se cargue correctamente con la configuración de `float32` para `whisperx`.